# Basic readout: aggregating `analytics` hourly buckets

There is no `GET /stats` endpoint yet (see `scw_js/analytics-implementation-plan.md`
— reads will be owner-signature-gated, like the Growth API). This notebook is a
throwaway prototype of the aggregation logic: list one **day's** hourly objects by
prefix, sum `hits`, merge `pages`. It also doubles as a rough draft for the future
growth-agent rollup (`growth-agent/website_analytics.json`, scoped separately) —
not production code, not wired into any endpoint.

**Day-prefix, not the bare site prefix**: `list_keys`/`listObjects` is one
unpaginated page (`shared/s3-utils/src/index.ts:280-286`), so listing the whole
`counts/fretchen.eu/` prefix would silently truncate past ~41 days of hourly objects.

In [ ]:
import os
from datetime import date, datetime, timezone

from dotenv import load_dotenv

from storage import LocalStorage, S3Storage

load_dotenv()

In [ ]:
def aggregate_day(storage, site: str, day: date) -> tuple[int, dict[str, int]]:
    """Sum hits and merge pages across one UTC day's hourly buckets."""
    prefix = f"counts/{site}/{day:%Y-%m-%d}T"
    total_hits = 0
    pages: dict[str, int] = {}
    for key in storage.list_keys(prefix):
        bucket = storage.read(key)
        if not bucket:
            continue
        total_hits += bucket.get("hits", 0)
        for path, count in bucket.get("pages", {}).items():
            pages[path] = pages.get(path, 0) + count
    return total_hits, pages

## 1. Fixture pass — validate the logic against known data

No credentials needed. Writes three synthetic hourly buckets, then checks
`aggregate_day` produces the hand-computed totals.

In [ ]:
local = LocalStorage()
fixture_day = date(2026, 8, 10)

local.write("counts/fixture-site/2026-08-10T00.json", {"hits": 5, "pages": {"/a": 3, "/b": 2}})
local.write("counts/fixture-site/2026-08-10T01.json", {"hits": 2, "pages": {"/a": 1, "/c": 1}})
local.write("counts/fixture-site/2026-08-10T02.json", {"hits": 4, "pages": {"/b": 4}})

fixture_hits, fixture_pages = aggregate_day(local, "fixture-site", fixture_day)
print(fixture_hits, fixture_pages)

assert fixture_hits == 11  # 5 + 2 + 4
assert fixture_pages == {"/a": 4, "/b": 6, "/c": 1}

## 2. Real readout

Will be small/empty until `01_smoke_test.ipynb` and the frontend wiring (PR2)
have generated real traffic.

In [ ]:
s3 = S3Storage(
    access_key=os.environ["SCW_ACCESS_KEY"],
    secret_key=os.environ["SCW_SECRET_KEY"],
)
today = datetime.now(timezone.utc).date()

total_hits, pages = aggregate_day(s3, "fretchen.eu", today)
print(f"{today}: {total_hits} hits across {len(pages)} distinct pages")

In [ ]:
# top pages, most-hit first
for path, count in sorted(pages.items(), key=lambda item: -item[1])[:10]:
    print(f"{count:>6}  {path}")